In [8]:
import os
import json
from pprint import pprint
from typing import Dict,Any

CONFIG_FILE = "products_config.json"
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as file:
        config_data:Dict = json.load(file)
        #pprint(config_data.get("project_name"))
        print(f"✅ 成功載入設定檔！專案名稱：{config_data.get('project_name')}")
        print(f"📦 監控品類數量：{len(config_data.get('monitor_products', []))} 大類")
else:
    print(f"❌ 找不到設定檔 {CONFIG_FILE}")



'毛寶企業產品與競品價格每日監控系統'
✅ 成功載入設定檔！專案名稱：毛寶企業產品與競品價格每日監控系統
📦 監控品類數量：5 大類


### 步驟 2：單一賣場抓取 — PChome 24h (API 方式)

In [16]:
from playwright.async_api import async_playwright, Browser,BrowserContext,APIResponse
from typing import Dict,Any
from pprint import pprint
import urllib.parse

async def fetch_pchome(context:BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 PChome 24h 購物抓取第一筆商品資訊 (API 方式)"""
    result = {
        "platform": "PChome 24h",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }
    encoded_kw= urllib.parse.quote(keyword)
    api_url = f"https://ecshweb.pchome.com.tw/search/v3.3/all/results?q={encoded_kw}&page=1"
    try:
        response:APIResponse = await context.request.get(api_url)
        if response.status == 200:
            data:Dict = await response.json()
            prods:list = data.get("prods",[])
            if prods:
                item:dict = prods[0]
                result['title'] = item.get("name", "未知的商品標題")
                result["price"] = int(item.get("price", 0))
                result["url"] = f"https://24h.pchome.com.tw/prod/{item.get('Id', '')}"
                result["status"] = "成功"
                return result
    except Exception as e:
        print(f"PChome 抓取失敗: {e}")
    return result
    
    
async with async_playwright() as p:
    browser:Browser = await p.firefox.launch(headless=True)
    context:BrowserContext = await browser.new_context()
    res:Dict[str, Any] = await fetch_pchome(context, "毛寶 洗衣槽")
    pprint(res)
    await browser.close()
    
    

{'platform': 'PChome 24h',
 'price': 559,
 'status': '成功',
 'title': '毛寶洗衣槽專用去汙劑  300gX1入X12盒  洗衣槽清潔劑',
 'url': 'https://24h.pchome.com.tw/prod/DEDG3L-A900J5MK0'}
